# Chromosomal enrichment of DEGs

Fisher's exact test (with Benjamini–Hochberg FDR) for enrichment of differentially expressed genes on each chromosome, as described in the manuscript Methods (`scipy.stats.fisher_exact`, `statsmodels.stats.multitest.multipletests`).

**Inputs** (tab-delimited; place under `inputs/` or set paths below):
- `Chromosome`, `Count` (DEGs on that chromosome), `TotalCount` (background genes on that chromosome)
- Optional density-aware table also needs `gene_density` and `len` / `lenMB`

Example files can be produced from DESeq2 results + a gene-position annotation (GRCm38.98).


In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# Relative to this notebook (bulk/chromosomal_enrichment/)
DATA_DIR = Path("inputs")  # change if needed
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
def chromosomal_enrichment(df: pd.DataFrame) -> pd.DataFrame:
    """Per-chromosome Fisher's exact test + BH FDR.

    Expects columns: Chromosome, Count, TotalCount
    """
    required = {"Chromosome", "Count", "TotalCount"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    total_deg = df["Count"].sum()
    total_bg = df["TotalCount"].sum()
    p_values = []
    for _, row in df.iterrows():
        a = int(row["Count"])
        b = int(row["TotalCount"] - row["Count"])
        c = int(total_deg - row["Count"])
        d = int(total_bg - total_deg - row["TotalCount"] + row["Count"])
        _, p = fisher_exact([[a, b], [c, d]])
        p_values.append(p)

    out = df.copy()
    out["P_Value"] = p_values
    out["FDR"] = multipletests(p_values, alpha=0.05, method="fdr_bh")[1]
    return out


## Standard enrichment (NPC / mESC examples)

Provide one table per contrast (e.g. all DEGs, UP only, DOWN only).


In [ ]:
# Example: NPC DEGs
npc_path = DATA_DIR / "gene_counts_NPC.txt"
if npc_path.exists():
    npc = pd.read_csv(npc_path, sep="\t")
    npc_res = chromosomal_enrichment(npc)
    npc_res.to_csv(OUT_DIR / "NPC_chromosomal_enrichment.csv", index=False)
    display(npc_res.sort_values("FDR"))
else:
    print(f"Place input table at {npc_path}")


In [ ]:
# Example: mESC DEGs
esc_path = DATA_DIR / "gene_counts_mESC.txt"
if esc_path.exists():
    esc = pd.read_csv(esc_path, sep="\t")
    esc_res = chromosomal_enrichment(esc)
    esc_res.to_csv(OUT_DIR / "mESC_chromosomal_enrichment.csv", index=False)
    display(esc_res.sort_values("FDR"))
else:
    print(f"Place input table at {esc_path}")


## Optional: gene-density–aware enrichment

Uses an expected count proportional to gene density × chromosome length.
Requires columns: `Chromosome`, `Count`, `TotalCount`, `gene_density`, `len`, `lenMB`.


In [ ]:
def chromosomal_enrichment_density(df: pd.DataFrame) -> pd.DataFrame:
    required = {"Chromosome", "Count", "TotalCount", "gene_density", "len", "lenMB"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    scaling = df["TotalCount"].sum() / (df["gene_density"] * df["lenMB"]).sum()
    df = df.copy()
    df["ExpectedCount"] = df["gene_density"] * df["len"] * scaling

    p_values = []
    for _, row in df.iterrows():
        contingency = [
            [row["Count"], row["TotalCount"] - row["Count"]],
            [row["ExpectedCount"], row["TotalCount"] * row["lenMB"] - row["ExpectedCount"]],
        ]
        _, p = fisher_exact(contingency)
        p_values.append(p)

    df["P_Value"] = p_values
    df["FDR"] = multipletests(p_values, alpha=0.05, method="fdr_bh")[1]
    return df


dens_path = DATA_DIR / "gene_counts_with_density.txt"
if dens_path.exists():
    dens = pd.read_csv(dens_path, sep="\t")
    dens_res = chromosomal_enrichment_density(dens)
    dens_res.to_csv(OUT_DIR / "chromosomal_enrichment_density.csv", index=False)
    display(dens_res.sort_values("FDR"))
else:
    print(f"Optional density table not found at {dens_path}")
